In [ ]:
# =========================================================
# CONFIGURAÇÃO LOCAL
# REMOVER ANTES DE SUBIR PARA O GITHUB
# =========================================================

import os

JAVA_HOME = (
    r"C:\Users\GCarapinadelima\Downloads"
    r"\microsoft-jdk-17.0.20.1-windows-x64"
    r"\jdk-17.0.20.1+1"
)

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = JAVA_HOME + r"\bin;" + os.environ["PATH"]


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]

GRAFICOS_DIR = PROJECT_ROOT / "scripts" / "Analytics" / "graficos" / "gold_07"
GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# CARREGAR GOLD 07
# ---------------------------------------------------------------------

caminho_gold_07 = (
    PROJECT_ROOT
    / "Gold"
    / "perguntas_negocio"
    / "gold_07_oportunidades_desafios"
)

arquivos_gold_07 = [
    str(arquivo) for arquivo in caminho_gold_07.glob("part-*.csv")
]

if not arquivos_gold_07:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_07}"
    )

df_gold_07 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_07)
)


# ---------------------------------------------------------------------
# FUNÇÕES AUXILIARES
# ---------------------------------------------------------------------

def para_pandas(df):
    return pd.DataFrame(
        [
            linha.asDict()
            for linha in df.collect()
        ]
    )


def salvar_grafico(nome):
    caminho = GRAFICOS_DIR / nome

    plt.savefig(
        caminho,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    print(f"Gráfico salvo: {caminho}")


labels = {
    "remuneracao_salario":
        "Remuneração / salário",

    "flexibilidade_de_trabalho_remoto":
        "Flexibilidade de trabalho remoto",

    "plano_de_carreira_e_oportunidades_de_crescimento":
        "Plano de carreira e crescimento",

    "beneficios":
        "Benefícios",

    "oportunidade_de_aprendizado_e_trabalhar_com_referencias":
        "Aprendizado e referências",

    "dividir_o_tempo_entre_entregas_tecnicas_e_gestao":
        "Conciliar entregas técnicas e gestão",

    "gerenciar_a_expectativa_das_areas":
        "Gerenciar expectativas das áreas",

    "gestao_de_projetos_envolvendo_areas_multidisciplinares":
        "Gestão de projetos multidisciplinares",

    "gerar_valor_para_as_areas_de_negocios":
        "Gerar valor para o negócio",

    "organizar_as_informacoes_com_qualidade_e_confiabilidade":
        "Qualidade e confiabilidade dos dados",

    "falta_de_expertise_ou_falta_de_recursos":
        "Falta de expertise ou recursos",

    "dados_da_empresa_nao_estao_prontos_para_uso_de_ia_generativa":
        "Dados não preparados para IA",

    "falta_de_compreensao_dos_casos_de_uso":
        "Falta de compreensão dos casos de uso",

    "retorno_sobre_investimento_roi_nao_comprovado_de_ia_generativa":
        "ROI da IA não comprovado",

    "preocupacoes_com_seguranca_e_privacidade_de_dados":
        "Segurança e privacidade dos dados"
}


# ---------------------------------------------------------------------
# PRINCIPAIS CRITÉRIOS PARA ESCOLHA DE UM EMPREGO
# ---------------------------------------------------------------------

df_criterios = (
    df_gold_07
    .filter(
        (F.col("edicao") == "2025-2026")
        &
        (
            F.col("categoria")
            == "criterios_para_escolher_emprego"
        )
    )
    .orderBy(F.desc("pct_adocao"))
    .limit(5)
)

pdf_criterios = para_pandas(df_criterios)

pdf_criterios["label"] = (
    pdf_criterios["opcao"]
    .map(labels)
    .fillna(pdf_criterios["opcao"])
)

pdf_criterios = (
    pdf_criterios
    .sort_values(
        "pct_adocao",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(12, 7))

barras = ax.barh(
    pdf_criterios["label"],
    pdf_criterios["pct_adocao"]
)

ax.bar_label(
    barras,
    fmt="%.1f%%",
    padding=5,
    fontsize=11
)

ax.set_title(
    "Principais critérios para escolha de um emprego\n"
    "Top 5 critérios entre profissionais de dados | 2025–2026",
    loc="left",
    fontsize=16,
    pad=18
)

ax.set_xlabel("Percentual de respondentes (%)")
ax.set_ylabel("")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlim(
    0,
    max(pdf_criterios["pct_adocao"]) * 1.15
)

plt.tight_layout()

salvar_grafico(
    "01_criterios_escolha_emprego.png"
)


# ---------------------------------------------------------------------
# PRINCIPAIS DESAFIOS DOS GESTORES
# ---------------------------------------------------------------------

df_gestores = (
    df_gold_07
    .filter(
        (F.col("edicao") == "2025-2026")
        &
        (
            F.col("categoria")
            == "desafios_como_gestor"
        )
    )
    .orderBy(F.desc("pct_adocao"))
    .limit(5)
)

pdf_gestores = para_pandas(df_gestores)

pdf_gestores["label"] = (
    pdf_gestores["opcao"]
    .map(labels)
    .fillna(pdf_gestores["opcao"])
)

pdf_gestores = (
    pdf_gestores
    .sort_values(
        "pct_adocao",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(12, 7))

barras = ax.barh(
    pdf_gestores["label"],
    pdf_gestores["pct_adocao"]
)

ax.bar_label(
    barras,
    fmt="%.1f%%",
    padding=5,
    fontsize=11
)

ax.set_title(
    "Principais desafios enfrentados pelos gestores de Dados\n"
    "Top 5 desafios reportados | 2025–2026",
    loc="left",
    fontsize=16,
    pad=18
)

ax.set_xlabel("Percentual de respondentes (%)")
ax.set_ylabel("")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlim(
    0,
    max(pdf_gestores["pct_adocao"]) * 1.18
)

plt.tight_layout()

salvar_grafico(
    "02_desafios_gestores.png"
)


# ---------------------------------------------------------------------
# PRINCIPAIS BARREIRAS PARA ADOÇÃO DE IA
# ---------------------------------------------------------------------

df_barreiras = (
    df_gold_07
    .filter(
        (F.col("edicao") == "2025-2026")
        &
        (
            F.col("categoria")
            == "motivos_para_nao_usar_ia"
        )
    )
    .orderBy(F.desc("pct_adocao"))
    .limit(5)
)

pdf_barreiras = para_pandas(df_barreiras)

pdf_barreiras["label"] = (
    pdf_barreiras["opcao"]
    .map(labels)
    .fillna(pdf_barreiras["opcao"])
)

pdf_barreiras = (
    pdf_barreiras
    .sort_values(
        "pct_adocao",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(12, 7))

barras = ax.barh(
    pdf_barreiras["label"],
    pdf_barreiras["pct_adocao"]
)

ax.bar_label(
    barras,
    fmt="%.1f%%",
    padding=5,
    fontsize=11
)

ax.set_title(
    "Principais barreiras para adoção de Inteligência Artificial\n"
    "Top 5 barreiras apontadas pelos profissionais | 2025–2026",
    loc="left",
    fontsize=16,
    pad=18
)

ax.set_xlabel("Percentual de respondentes (%)")
ax.set_ylabel("")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlim(
    0,
    max(pdf_barreiras["pct_adocao"]) * 1.18
)

plt.tight_layout()

salvar_grafico(
    "03_barreiras_ia.png"
)


# ---------------------------------------------------------------------
# EVOLUÇÃO DAS PRINCIPAIS BARREIRAS PARA ADOÇÃO DE IA
# ---------------------------------------------------------------------

principais_barreiras = [
    "falta_de_expertise_ou_falta_de_recursos",
    "dados_da_empresa_nao_estao_prontos_para_uso_de_ia_generativa",
    "falta_de_compreensao_dos_casos_de_uso",
    "retorno_sobre_investimento_roi_nao_comprovado_de_ia_generativa",
    "preocupacoes_com_seguranca_e_privacidade_de_dados"
]

df_historico_ia = (
    df_gold_07
    .filter(
        (F.col("categoria") == "motivos_para_nao_usar_ia")
        &
        F.col("opcao").isin(principais_barreiras)
    )
    .select(
        "edicao",
        "opcao",
        "pct_adocao"
    )
    .orderBy(
        "edicao",
        "opcao"
    )
)

pdf_historico_ia = para_pandas(df_historico_ia)

pdf_historico_ia["label"] = (
    pdf_historico_ia["opcao"]
    .map(labels)
    .fillna(pdf_historico_ia["opcao"])
)

ordem_edicoes = [
    "2023-2024",
    "2024-2025",
    "2025-2026"
]

pdf_historico_ia["edicao"] = pd.Categorical(
    pdf_historico_ia["edicao"],
    categories=ordem_edicoes,
    ordered=True
)

pdf_historico_ia = (
    pdf_historico_ia
    .sort_values("edicao")
)

fig, ax = plt.subplots(figsize=(13, 8))

for label, grupo in pdf_historico_ia.groupby(
    "label",
    observed=True
):
    grupo = grupo.sort_values("edicao")

    ax.plot(
        grupo["edicao"],
        grupo["pct_adocao"],
        marker="o",
        linewidth=2.2,
        label=label
    )

    for _, linha in grupo.iterrows():
        ax.annotate(
            f'{linha["pct_adocao"]:.1f}%',
            (
                linha["edicao"],
                linha["pct_adocao"]
            ),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            fontsize=9
        )

ax.set_title(
    "Evolução das principais barreiras para adoção de IA\n"
    "Comparação entre as três edições da State of Data Brasil",
    loc="left",
    fontsize=16,
    pad=18
)

ax.set_xlabel("")
ax.set_ylabel("Percentual de respondentes (%)")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    title="Barreira",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()

salvar_grafico(
    "04_evolucao_barreiras_ia.png"
)